# Phase 5-C1 — Final Agent Integration & Authorization Readiness

[Open in Colab](https://colab.research.google.com/github/hh4832/strategy-research-agent/blob/feature/phase5c1-first-scientific-research/notebooks/Phase5C1_Long_Development_Readiness.ipynb)

Edit human inputs only in the first code cell. `PREPARE_ONLY` is the default: it verifies Core and schedule pins, stages the existing recovery archive locally, renews the receipt, performs full non-consuming preflight, checks all four fixed cost scenarios, writes the research proposal and authorization draft, and archives the evidence to Drive. It does not create authorization, post approvals, execute the strategy, or consume a batch.

The dated schedule and historical SEC/FINRA source records are pinned to the Core delivery. SEC New York Fill-date mapping remains a disclosed `RESEARCH_ONLY` assumption, not broker-actual. The existing formal Market Gate stays `NO_GO`.


In [ ]:
# ALL HUMAN SETTINGS / APPROVAL INPUTS — edit only this cell.
import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive, userdata
drive.mount('/content/drive')

AGENT_REPOSITORY = 'https://github.com/hh4832/strategy-research-agent.git'
AGENT_BRANCH = 'feature/phase5c1-first-scientific-research'
AGENT_SHA = '9e92c0f4f35f909aec93dbab70946b0174b599c4'  # CI-tested Agent CODE pin; never branch HEAD
NOTEBOOK_DELIVERY_COMMIT = None  # optional: delivery commit from this Notebook URL; separate from AGENT_SHA
CORE_REPOSITORY = 'https://github.com/hh4832/market-research-core.git'
CORE_SHA = '50ea3bfa42df7708dd3a1b21f17798c206be054e'
SCHEDULE_SHA256 = '43b6045758760a40ad88d741b1a97a73cfb0b6cf6b2a4a7f1f3b31d5085a4216'
MODE = 'PREPARE_ONLY'  # PREPARE_ONLY or EXECUTE_AUTHORIZED
SNAPSHOT_ID = 'TSM_DEV_1M_20161019_20241231_ca9f87df83f31bd24211'
RECOVERY_ARCHIVE = Path('/content/drive/MyDrive/00Quant_Research/data/development_recovery_archives/TSM_DEV_1M_20161019_20241231_ca9f87df83f31bd24211_20261110T142034_335006Z')
ORIGINAL_APPROVAL_EVIDENCE_PATH = RECOVERY_ARCHIVE / 'recovery_provenance.json'
APPROVAL_IS_IN_RECOVERY_RECORD = True
ORIGINAL_CONVERSION_APPROVAL_REF = 'https://github.com/hh4832/market-research-core/pull/4#issuecomment-6768321310'

# OBSERVED values are historical observations only. They never become reviewed pins automatically.
OBSERVED_DATA_PINS = {
    'marker_sha256':'8a980bd92045da59d573b249162ba5dce581a1c74802663b0b02cbb0756ee7a5',
    'recovery_provenance_sha256':'dddc8794c4bf52556c12d5c919e779d9bb3b7aba2770e73f34c838fe5ab8b6aa',
    'binding_sha256':'9822ad8db3bdde056c0192cb056577bdc91485af92f45feb47d1246f5c6a8f1f',
    'manifest_sha256':'edabc0abf38bab6cad013b2248b68344dc0073da105a6085dd5e8f05f85272e4',
    'lineage_sha256':'baeac6bc2e794038646ee7b313edc27038090862ea0da73cc64a87e87a9adffa',
    'parquet_sha256':'e1e7955401fcbe4a84e7486fa7b1cc8ae97a92513426086cf7902afcf958ee03'}
# First run: None. After a person independently reviews all six values, paste all six here and cite the review evidence.
REVIEWED_DATA_PINS = None
PIN_REVIEW_EVIDENCE_REF = ''

NEW_AUTHORIZATION_ID = 'RA-IBKR_TSM-SMA520_P5C1-0001'
EXECUTOR_ID = 'colab-hh4832-exec-01'
GOVERNANCE_ROOT = None  # None means the pinned Agent checkout
PREVIOUS_PREPARATION_DIRECTORY = None  # governed B3-C prepare directory on Drive for EXECUTE_AUTHORIZED
BIND_AUTHORIZATION_URL = ''  # personal GitHub approval comment URL
APPROVE_TASK_URL = ''        # separate personal GitHub approval comment URL
LOCAL_PARENT = Path('/content/phase5c1')
DRIVE_OUTPUT_PARENT = Path('/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1')
CONFIRM_SETTINGS = False  # personally review and set True

import re
assert MODE in ('PREPARE_ONLY','EXECUTE_AUTHORIZED')
assert CONFIRM_SETTINGS, 'STOP: review and confirm this first cell; no strategy execution or batch consumption'
assert CORE_SHA == '50ea3bfa42df7708dd3a1b21f17798c206be054e' and SCHEDULE_SHA256 == '43b6045758760a40ad88d741b1a97a73cfb0b6cf6b2a4a7f1f3b31d5085a4216'
assert RECOVERY_ARCHIVE.is_dir() and ORIGINAL_APPROVAL_EVIDENCE_PATH.is_file()
assert LOCAL_PARENT.resolve().is_relative_to(Path('/content').resolve()) and not LOCAL_PARENT.resolve().is_relative_to(Path('/content/drive').resolve())
assert DRIVE_OUTPUT_PARENT.resolve().is_relative_to(Path('/content/drive').resolve())
if REVIEWED_DATA_PINS is not None:
    assert set(REVIEWED_DATA_PINS)==set(OBSERVED_DATA_PINS) and REVIEWED_DATA_PINS==OBSERVED_DATA_PINS, 'STOP: all six pins must match the observed values exactly'
    assert PIN_REVIEW_EVIDENCE_REF, 'STOP: independently reviewed pin evidence reference required'
if MODE == 'EXECUTE_AUTHORIZED':
    assert REVIEWED_DATA_PINS is not None, 'STOP: independently reviewed pins required'
    if PREVIOUS_PREPARATION_DIRECTORY and BIND_AUTHORIZATION_URL and APPROVE_TASK_URL:
        assert BIND_AUTHORIZATION_URL.startswith('https://github.com/') and APPROVE_TASK_URL.startswith('https://github.com/')
        assert BIND_AUTHORIZATION_URL != APPROVE_TASK_URL
RUN_ID = datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S_%f')
LOCAL_SESSION = LOCAL_PARENT / RUN_ID
DRIVE_OUTPUT = DRIVE_OUTPUT_PARENT / RUN_ID
assert not LOCAL_SESSION.exists() and not DRIVE_OUTPUT.exists(), 'STOP: outputs must be new and immutable'
print('Mode:',MODE,'| snapshot:',SNAPSHOT_ID,'| Core:',CORE_SHA,'| dated schedule:',SCHEDULE_SHA256)
print('Existing recovery archive:',RECOVERY_ARCHIVE)


In [ ]:
# Pinned, clean source setup. The Notebook delivery commit may differ from the tested Agent CODE pin.
import subprocess,sys,importlib,shutil
assert sys.version_info >= (3,11)
def run(*args): return subprocess.run(args,check=True,text=True,capture_output=True).stdout.strip()
assert AGENT_SHA == '9e92c0f4f35f909aec93dbab70946b0174b599c4'
AGENT_ROOT=Path('/content/strategy-research-agent-'+AGENT_SHA)
CORE_ROOT=Path('/content/market-research-core-'+CORE_SHA)
if not AGENT_ROOT.exists(): run('git','clone','--branch',AGENT_BRANCH,'--single-branch',AGENT_REPOSITORY,str(AGENT_ROOT))
assert run('git','-C',str(AGENT_ROOT),'status','--porcelain')==''
run('git','-C',str(AGENT_ROOT),'checkout','--detach',AGENT_SHA)
FETCHED_AGENT_SHA=run('git','-C',str(AGENT_ROOT),'rev-parse','HEAD')
assert FETCHED_AGENT_SHA == AGENT_SHA, 'STOP: Agent CODE pin mismatch'
assert run('git','-C',str(AGENT_ROOT),'status','--porcelain')==''
if not CORE_ROOT.exists():
    run('git','clone',CORE_REPOSITORY,str(CORE_ROOT))
    run('git','-C',str(CORE_ROOT),'checkout','--detach',CORE_SHA)
assert run('git','-C',str(CORE_ROOT),'rev-parse','HEAD')==CORE_SHA
assert run('git','-C',str(CORE_ROOT),'status','--porcelain')==''
subprocess.run([sys.executable,'-m','pip','install','-e',str(CORE_ROOT),'-e',str(AGENT_ROOT)+'[research]'],check=True)
sys.path[:0]=[str(AGENT_ROOT/'src'),str(AGENT_ROOT),str(CORE_ROOT/'src')]
importlib.invalidate_caches()
from strategy_agent.execution import c1_integration as C
from strategy_agent.adapters.mrc_phase4a import core_repository_root
assert C.CORE==CORE_SHA and C.SCHEDULE_SHA256==SCHEDULE_SHA256 and C.SNAPSHOT==SNAPSHOT_ID
assert C.B.EXECUTOR==EXECUTOR_ID
print('Pinned Agent HEAD:',FETCHED_AGENT_SHA,'| Core:',CORE_SHA,'| Notebook delivery commit:',NOTEBOOK_DELIVERY_COMMIT)


In [ ]:
# Real archive checks, local staging, renewed receipt, full Core preflight and cost schedule compatibility.
from strategy_agent.adapters.mrc_phase4a import core_repository_root
SCHEDULE_SHA256=C.SCHEDULE_SHA256
evidence=json.loads(ORIGINAL_APPROVAL_EVIDENCE_PATH.read_text())
approval=evidence['conversion_approval'] if APPROVAL_IS_IN_RECOVERY_RECORD else evidence
assert C.is_original_approval_url(ORIGINAL_CONVERSION_APPROVAL_REF) and C.is_original_approval_url(approval['approval_ref'])
inspection=C.inspect_archive(RECOVERY_ARCHIVE,conversion_approval=approval,trusted_pins=REVIEWED_DATA_PINS)
LOCAL_SESSION.mkdir(parents=True,exist_ok=False)
staged=C.stage_archive(inspection,LOCAL_SESSION/'staged')
schedules=C.load_dated_schedules(core_repository_root(),expected_sha256=SCHEDULE_SHA256)
cost_check=C.validate_cost_scenario_compatibility(AGENT_ROOT,schedules)
assert cost_check['status']=='PASS' and set(schedules)=={'BASE','LOW','STRESS','CAT_STRESS'}
print('Snapshot pin status:',inspection['status'],'| receipt renewal:',staged['preflight']['status'])
print('Observed pins (not independently reviewed):',json.dumps(inspection['observed_pins'],indent=2))
print('Cost coverage: 2017-01-01 through 2024-12-31 | four scenarios:',cost_check['scenarios'])
print('SEC:',cost_check['official_sec_rates'],'| FINRA TAF:',cost_check['official_finra_taf_rates'])
print('SEC Fill-date mapping:',cost_check['sec_date_convention'],'| TAF/CAT:',cost_check['taf_cat_assumption'])
print('Formal Market Gate remains NO_GO.')


In [ ]:
# PREPARE_ONLY writes only the research proposal/readiness archive. EXECUTE_AUTHORIZED remains on B3-C REAL_GATED.
OUTPUT=LOCAL_SESSION/'preparation'
OUTPUT.mkdir(parents=True,exist_ok=False)
DRIVE_OUTPUT.mkdir(parents=True,exist_ok=False)
result=None
if MODE=='PREPARE_ONLY':
    result=C.prepare(staged,OUTPUT/'draft',agent_commit=FETCHED_AGENT_SHA,root=AGENT_ROOT)
    print(json.dumps(result,indent=2))
else:
    gov=Path(GOVERNANCE_ROOT) if GOVERNANCE_ROOT else AGENT_ROOT
    if not (PREVIOUS_PREPARATION_DIRECTORY and BIND_AUTHORIZATION_URL and APPROVE_TASK_URL):
        # Authorization must already exist. This creates the standard B3-C request and approval lines only.
        summary=C.prepare(staged,OUTPUT/'draft',agent_commit=FETCHED_AGENT_SHA,root=AGENT_ROOT)
        prepared=C.prepare_governed_request(OUTPUT/'draft',governance_root=gov,agent_commit=FETCHED_AGENT_SHA)
        print('B3-C request prepared; no execution or batch was attempted.')
        print('BIND_AUTHORIZATION:',prepared['BIND_AUTHORIZATION'])
        print('APPROVE_TASK:',prepared['APPROVE_TASK'])
        print('Set PREVIOUS_PREPARATION_DIRECTORY to',DRIVE_OUTPUT/'draft'/'governed_request','personally post the two distinct GitHub approvals, then rerun with its Drive path and URLs.')
    else:
        from strategy_agent.approval import production_verifier
        import os
        TOKEN=userdata.get('GITHUB_TOKEN')
        assert TOKEN,'Colab Secret GITHUB_TOKEN is required for live approval verification'
        os.environ['GITHUB_TOKEN']=TOKEN
        stamp=datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S')
        # Reuse the exact C1 preparation and B3-C request reviewed before the GitHub approvals.
        governed_request=Path(PREVIOUS_PREPARATION_DIRECTORY)
        p5prep=governed_request.parent
        summary=json.loads((p5prep/'readiness.json').read_text())
        assert summary['data_binding']=='REVIEWED_PINS_VERIFIED' and summary['cost_schedule_sha256']==SCHEDULE_SHA256
        verifier=production_verifier(); assert verifier.production
        backend=C.B.production_backend(root=gov,runner_root=Path(summary['artifact_root']),governance_root=gov,
            snapshot_root=Path(staged['local_root']),session_dir=DRIVE_OUTPUT/'sessions'/stamp,git_commit=FETCHED_AGENT_SHA,
            context={'notebook':'PHASE5C1_REAL_GATED','schedule_sha256':SCHEDULE_SHA256},
            canonical_config=Path(summary['config_path']),authorization_id=NEW_AUTHORIZATION_ID)
        result=C.execute_authorized(p5prep,governance_root=gov,prepared_dir=governed_request,
            bind_url=BIND_AUTHORIZATION_URL,task_url=APPROVE_TASK_URL,verifier=verifier,backend=backend,
            work_dir=Path('/content/phase5c1_run'),claim_dir=DRIVE_OUTPUT/'claims')
    if result is not None: print(json.dumps(result,indent=2,default=str))
# Complete, immutable Drive evidence copy. Never copy the staged Parquet into the evidence archive.
C.save(OUTPUT/'notebook_context.json',{'agent_commit':FETCHED_AGENT_SHA,'core_commit':CORE_SHA,'schedule_sha256':SCHEDULE_SHA256,
    'mode':MODE,'snapshot_id':SNAPSHOT_ID,'pin_review_evidence_ref':PIN_REVIEW_EVIDENCE_REF,
    'agent_code_pin':FETCHED_AGENT_SHA,'notebook_delivery_commit':globals().get('NOTEBOOK_DELIVERY_COMMIT'),
    'pin_status':inspection['status'],'observed_pins':inspection['observed_pins'],
    'strategy_execution':'NOT_EXECUTED' if MODE=='PREPARE_ONLY' or not (PREVIOUS_PREPARATION_DIRECTORY and BIND_AUTHORIZATION_URL and APPROVE_TASK_URL) else 'B3C_REAL_GATED',
    'consumed_batches':0 if MODE=='PREPARE_ONLY' or not (PREVIOUS_PREPARATION_DIRECTORY and BIND_AUTHORIZATION_URL and APPROVE_TASK_URL) else 'GOVERNED_LEDGER',
    'authorization_created':False,'formal_market_gate':'NO_GO'})
# Audit-only backup of this runtime's staging metadata; never a reusable receipt.
audit_root=DRIVE_OUTPUT/'runtime_audit'
audit_root.mkdir(parents=True,exist_ok=False)
audit_hashes={}
for audit_name in ('data_preflight.json','runtime_binding.json','pin_summary.json'):
    audit_source=LOCAL_SESSION/'staged'/audit_name
    audit_target=audit_root/audit_name
    source_sha_before=C.sha(audit_source)
    with audit_source.open('rb') as reader,audit_target.open('xb') as writer:
        shutil.copyfileobj(reader,writer)
    source_sha_after=C.sha(audit_source)
    archive_sha=C.sha(audit_target)
    assert source_sha_before == source_sha_after == archive_sha, 'STOP: runtime audit backup SHA mismatch: '+audit_name
    audit_hashes[audit_name]={'source_sha256_before':source_sha_before,
        'source_sha256_after':source_sha_after,'archive_sha256':archive_sha}
C.save(audit_root/'AUDIT_ONLY.json',{'purpose':'AUDIT_ONLY_NOT_REUSABLE_RUNTIME_RECEIPT',
    'runtime_binding_reusable':False,'receipt_policy':'Renew the receipt and run full Core preflight in every new runtime; never use this archived binding for execution.',
    'files':audit_hashes,'agent_code_pin':FETCHED_AGENT_SHA,'core_commit':CORE_SHA})
shutil.copytree(OUTPUT,DRIVE_OUTPUT/'notebook_output')
index={str(p.relative_to(DRIVE_OUTPUT)):C.sha(p) for p in sorted(DRIVE_OUTPUT.rglob('*')) if p.is_file()}
C.save(DRIVE_OUTPUT/'ARCHIVE_INDEX.json',{'files_sha256':index,'agent_commit':FETCHED_AGENT_SHA,'core_commit':CORE_SHA,'schedule_sha256':SCHEDULE_SHA256,'timezone':'Asia/Taipei'})
print('Drive evidence archive:',DRIVE_OUTPUT)


## Authorization and run boundary

The first `PREPARE_ONLY` run reports the six data pins as `OBSERVED_NOT_TRUSTED`. A person must compare all six pins independently and cite that review before filling `REVIEWED_DATA_PINS`; the notebook never promotes observed pins. The official historical SEC and FINRA TAF source records and canonical Schedule SHA are checked before readiness is reported. SEC New York Fill-date remains a predeclared research assumption, not broker actual. The conservative TAF/CAT BOTH-side, no-cap assumption remains unchanged.

`EXECUTE_AUTHORIZED` enters the existing B3-C request, binding, personal GitHub approval, `REAL_GATED` A01–A10, atomic one-Batch ledger, and ResearchOnlyRunner flow. The official schedule SHA must also be present in the formal authorization. PREPARE steps do not create an authorization, post comments, run SMA, read Holdout, or consume a Batch. The formal Market Gate stays `NO_GO`.

The Agent code is checked out at the fixed CI-tested `AGENT_SHA`, independently of the Notebook delivery commit. `runtime_audit/` archives the three local staging verification files with source-before/source-after/Drive-read-back SHA256 checks. Its `runtime_binding.json` is audit evidence only, never a receipt reusable in another Runtime; every new Runtime must renew its own receipt and pass full Core preflight.
